In [2]:
import sys
import os
import numpy as np
from tqdm import tqdm
import torch

sys.path.append(os.path.abspath(".."))

from gymnasium.vector import SyncVectorEnv
from core.env.core import SnakeEnv
from core.env.types import ObserveType, RewardOptions

num_envs, total_episodes = 16, 100000
env = SyncVectorEnv(
    [
        lambda i=i: SnakeEnv(
            width=20,
            height=20,
            obs_type=ObserveType.VEC_11,
            num_apples=3,
            num_obstacles=15,
            seed=42 + i,
            reward_options=RewardOptions(
                eats_apple=24.0,
                penalty_step=-0.01,
                penalty_loop=-0.1,
                death_wall=-20.0,
                death_self=-20.0,
                shaping_closer=0.1,
                shaping_further=-0.1,
                complete=100.0,
            ),
        )
        for i in range(num_envs)
    ]
)

epsilon_decay = (0.01) ** (1 / (total_episodes * 0.6))

training_logs = []
episode_rewards = np.zeros(num_envs)
completed = 0
states, infos = env.reset()
best_reward = -np.inf

In [6]:
from agents.actor_critic import ActorCriticAgent
# Configuration
agent_name = "actor_critic_snake"

# Initialize Actor-Critic Agent
# Uses state_dim=11 for the raw bits instead of 2048 for the table index
agent = ActorCriticAgent(
    state_dim=11, 
    action_dim=3, 
    lr=0.0003, # Deep RL usually needs a smaller learning rate than Tabular Q-Learning
    gamma=0.99,
    seed=42
)


AttributeError: partially initialized module 'torch._dynamo' from 'd:\Work\snake-rl\.venv\Lib\site-packages\torch\_dynamo\__init__.py' has no attribute 'utils' (most likely due to a circular import)

In [ ]:
with tqdm(total=total_episodes, desc="Parallel Actor-Critic Training") as pbar:
    while completed < total_episodes:
        # 1. Get actions from the Actor (Policy)
        actions = [int(agent.act(s)) for s in states]
        
        # 2. Step the vector environment
        next_states, rewards, terminated, truncated, next_infos = env.step(actions)
        
        for i in range(len(actions)):
            s = states[i]
            a = actions[i]
            r = float(rewards[i])
            ns = next_states[i]
            done_i = bool(terminated[i])
            trunc_i = bool(truncated[i])

            # 3. Update the Actor and Critic networks
            # This handles the log_prob and state_value internally from the last act() call
            agent.update(s, a, r, ns, done_i)
            episode_rewards[i] += r

            if done_i or trunc_i:
                completed += 1
                if completed <= total_episodes:
                    pbar.update(1)
                    
                    # Log data
                    training_logs.append({
                        "episode": completed,
                        "reward": float(episode_rewards[i]),
                    })

                    # Track best performance and save .pth instead of .pkl
                    if len(training_logs) >= 100:
                        recent_avg = np.mean([log["reward"] for log in training_logs[-100:]])
                        if recent_avg > best_reward:
                            best_reward = recent_avg
                            agent.save(f"../artifacts/models/{agent_name}_best.pth")

                    # Console logging every 5000 episodes
                    if completed % 5000 == 0:
                        recent_avg = (
                            np.mean([log["reward"] for log in training_logs[-100:]])
                            if len(training_logs) >= 100
                            else float(episode_rewards[i])
                        )
                        tqdm.write(
                            f"Ep {completed}/{total_episodes} | Avg Reward (100): {recent_avg:.2f} | Best Avg: {best_reward:.2f}"
                        )
                
                # Reset reward counter for the next episode in this environment slot
                episode_rewards[i] = 0.0

        # Update current states
        states = next_states

env.close()